In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [8]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [9]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [10]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="openai",
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("OPENAI_API_KEY"),
    model=os.environ.get("OPENAI_MODEL"),
    temperature=0.3,
)


agent = create_agent(
    model=model,
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [12]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database. Could you tell me:

- Which database type you’re using (SQL like PostgreSQL/MySQL/SQLite, or a NoSQL like MongoDB)?
- The name of the table/collection (likely artists)?
- Whether you want to include inactive/soft-deleted records or only active ones?

In the meantime, here are ready-to-run queries for common setups:

SQL (e.g., PostgreSQL, MySQL, SQLite)
- All artists:
  SELECT COUNT(*) AS total_artists FROM artists;
- Active/not-deleted (if you track this with a deleted_at column or is_active flag):
  SELECT COUNT(*) AS total_artists FROM artists WHERE deleted_at IS NULL;
  -- or
  SELECT COUNT(*) AS total_artists FROM artists WHERE is_active = TRUE;

MongoDB
- All artists:
  db.artists.countDocuments({});
- Active/not-deleted (if you have an active field):
  db.artists.countDocuments({ active: true });

ORM examples (if you’re using a language/framework)
- Python with Django:
  Artist.objects.count()  -- or Artist.objects.filter(active=True).count